In [1]:
import os
from dotenv import load_dotenv

os.environ['ANONYMIZED_TELEMETRY'] = 'False'    # silence Chroma telemetry

from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain.chains import create_history_aware_retriever
from langchain_core.messages import HumanMessage, AIMessage
from langchain_community.document_compressors import FlashrankRerank
from flashrank import Ranker
from langchain.retrievers import ContextualCompressionRetriever

load_dotenv()
print('Imports ready')

Imports ready


Load the domain‑tagged vector store

In [2]:
from pathlib import Path

# Absolute path so it works regardless of where the notebook runs from
persist_dir = r'C:\Users\USER\rag_course\chroma_db_domain'

# Load the store
embeddings = OpenAIEmbeddings()
vectorstore = Chroma(
    persist_directory=persist_dir,
    embedding_function=embeddings
)

# Base retriever — retrieve MORE candidates(for reranking)
base_retriever = vectorstore.as_retriever(search_kwargs={'k': 20})

print(f'Vector store loaded')
print(f'   Chunks in store: {vectorstore._collection.count()}')
print(f'   Retrieval k: 20 (candidates for reranking)')

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Vector store loaded
   Chunks in store: 80
   Retrieval k: 20 (candidates for reranking)


Build the FlashRank reranker

In [3]:
# Load the local FlashRank model 
ranker = Ranker(model_name='ms-marco-MiniLM-L-12-v2')

# Wrap it in LangChain's reranker
reranker = FlashrankRerank(
    model='ms-marco-MiniLM-L-12-v2',
    top_n=5
)

# Combine base retriever + reranker into one retriever
reranking_retriever = ContextualCompressionRetriever(
    base_compressor=reranker,
    base_retriever=base_retriever
)

print('Reranker ready')
print('   Retrieval: 20 candidates → reranked → top 5')

Reranker ready
   Retrieval: 20 candidates → reranked → top 5


Compare retrieval before vs. after reranking

In [4]:
query = 'What are the top causes of death in Nigeria?'

# --- Plain retrieval (top 5 by similarity) ---
plain_docs = base_retriever.invoke(query)[:5]

print('PLAIN RETRIEVAL (top 5 by similarity)')
print('=' * 70)

for i, d in enumerate(plain_docs):
    domain = d.metadata.get('domain', '?')
    print(f'[{i}] domain={domain}')
    
    print('     ' + d.page_content[:180].replace('\n', ' '))
    print()

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


PLAIN RETRIEVAL (top 5 by similarity)
[0] domain=health
     infectious diseases, sewage disposal, health insurance,  water supply, air pollution, noise pollution, environmen- tal radiation, housing, solid waste disposal, disaster  managemen

[1] domain=health
      The top causes of death in Nigeria are; malaria,  lower respiratory infections, HIV/AIDS,       diarrheal diseases, road injuries, protein -energy  malnutrition, cancer, meningit

[2] domain=health
     health care services, brain drain, and irrational           appointment of health workers among others. A new  global burden has revealed that malaria and HIV are  still leading ca

[3] domain=health
     disease listed Nigeria and other developing countries as  the worst hit with deaths from non -communicable         diseases.5 These diseases with a rising burden in Nigeria  includ

[4] domain=health
     birth weight (2%), birth asphyxia and birth trauma  (2%) and maternal conditions (2%). Nevertheless the  effective publ

Reranking retriever

In [5]:
# --- Reranked retrieval (top 5 after FlashRank) ---
reranked_docs = reranking_retriever.invoke(query)

print('RERANKED RETRIEVAL (top 5 after FlashRank)')
print('=' * 70)

for i, d in enumerate(reranked_docs):
    domain = d.metadata.get('domain', '?')
    score = d.metadata.get('relevance_score', 'n/a')
    print(f'[{i}] domain={domain} score={score}')
    print('     ' + d.page_content[:200].replace('\n', ' '))
    print()

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


RERANKED RETRIEVAL (top 5 after FlashRank)
[0] domain=health score=0.9999481439590454
      The top causes of death in Nigeria are; malaria,  lower respiratory infections, HIV/AIDS,       diarrheal diseases, road injuries, protein -energy  malnutrition, cancer, meningitis, stroke and  tube

[1] domain=health score=0.9997609257698059
     infectious diseases, sewage disposal, health insurance,  water supply, air pollution, noise pollution, environmen- tal radiation, housing, solid waste disposal, disaster  management, control of vector

[2] domain=health score=0.9997542500495911
     disease listed Nigeria and other developing countries as  the worst hit with deaths from non -communicable         diseases.5 These diseases with a rising burden in Nigeria  include cardiovascular dis

[3] domain=health score=0.9993085861206055
     health care services, brain drain, and irrational           appointment of health workers among others. A new  global burden has revealed that malaria and HIV 

In [6]:
d

Document(metadata={'id': 4, 'relevance_score': 0.9661943, 'domain': 'health', 'file_name': 'nigeria_health_diseases_and_prevention.pdf', 'page': 2, 'source': 'C:\\Users\\USER\\rag_course\\04_data_ingestion_document_processing\\data\\nigeria_health_diseases_and_prevention.pdf'}, page_content='birth weight (2%), birth asphyxia and birth trauma \n(2%) and maternal conditions (2%). Nevertheless the \neffective public health interventions that could have \nprevented most of deaths exist, but coverage is low \ndue to weak and under -resourced health systems. \nSome of the weaknesses can be attributed to          \nchallenges related to leadership and governance; \nhealth workforce; medical products, vaccines and \ntechnologies; information; financing; and services \ndelivery.17 \n \nIn Nigeria communicable and infectious diseases are \nthe major health problem. 3 Nigeria has slowly       \nentered the era of ‘disease of the affluent ” Diseases \nlike Hypertension, Cancer, Obesity etc. Most o

In [7]:
for i, d in enumerate(reranked_docs):
    print(d.metadata.get('relevance_score'))

0.99994814
0.9997609
0.99975425
0.9993086
0.9661943


Build the prompts

In [8]:
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

# Prompt 1: Rewrite follow-up questions into standalone form
rewrite_prompt = ChatPromptTemplate.from_messages([
    ('system',
     'You rewrite a user\'s latest question into a standalone question using the chat history.\n'
     '\n'
     'Rules:\n'
     '1. If the latest question refers to something from the history (like "the first one", '
     '"that", "it", "the second option"), REPLACE that reference with the actual item from the history.\n'
     '2. Do NOT answer the question. Only rewrite it.\n'
     '3. If the question is already standalone, return it unchanged.\n'
     '\n'
     'Examples:\n'
     '- History mentions "Malaria" first, then "HIV/AIDS".\n'
     '  User: "Tell me more about the first one."\n'
     '  Rewritten: "Tell me more about malaria."\n'
     '\n'
     '- History mentions "Cassava Mosaic Disease" first, then "Maize Smut".\n'
     '  User: "How do I control the second one?"\n'
     '  Rewritten: "How do I control Maize Smut?"'),
    MessagesPlaceholder('chat_history'),
    ('human', '{input}'),
])

# Prompt 2: Answer using retrieved context
qa_prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a helpful assistant. '
               'Answer the question using only the provided context. '
               'If you don\'t know, say you don\'t know.'),
    ('human', 'Context:\n{context}\n\nQuestion: {input}'),
])

print('Prompts ready')

Prompts ready


Build the reusable chains

In [9]:
# Reusable chains — build once, invoke many times
rewrite_chain = rewrite_prompt | llm | StrOutputParser()
qa_chain = qa_prompt | llm | StrOutputParser()

print('Chains ready')

Chains ready


Helper functions and the reranked ask()

In [10]:
def rewrite_question(question, history):
    """Turn a follow-up into a standalone question using chat history."""
    return rewrite_chain.invoke({
        'chat_history': history,
        'input': question,
    })

def generate_answer(question, context):
    """Generate the final answer from retrieved context."""
    return qa_chain.invoke({
        'context': context,
        'input': question,
    })

# Session store and history retrieval
store = {}

def get_history(session_id):
    if session_id not in store:
        store[session_id] = []
    return store[session_id]

def ask(question, session_id):
    """Full conversational RAG with reranking."""

    history = get_history(session_id)

    # 1. Rewrite if history exists
    rewritten = rewrite_question(question, history) if history else question

    # 2. Retrieve top 5 candidates AFTER reranking
    docs = reranking_retriever.invoke(rewritten)

    # 3. Build the context
    context = '\n\n'.join(d.page_content for d in docs)

    # 4. Generate the answer
    answer = generate_answer(rewritten, context)

    # 5. Save the turn to history
    history.append(HumanMessage(content=question))
    history.append(AIMessage(content=answer))

    return answer

print('Reranked conversational helper ready')

Reranked conversational helper ready


In [11]:
session_id = 'rerank-test-1'
q1 = 'What are the top causes of death in Nigeria?'
a1 = ask(q1, session_id)

print('👤 User:', q1)
print('🤖 Assistant:', a1)
print('-' * 70)

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


👤 User: What are the top causes of death in Nigeria?
🤖 Assistant: The top causes of death in Nigeria are:

1. Malaria
2. Lower Respiratory Infections
3. HIV/AIDS
4. Diarrheal Diseases
5. Road Injuries
6. Protein-energy malnutrition
7. Cancer
8. Meningitis
9. Stroke
10. Tuberculosis
----------------------------------------------------------------------


Turn 2 — vague follow‑up

In [12]:
# Same session_id → same history
q2 = 'Tell me more about the first one.'
a2 = ask(q2, session_id)

print('👤 User:', q2)
print('🤖 Assistant:', a2)
print('-' * 70)

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


👤 User: Tell me more about the first one.
🤖 Assistant: Malaria remains the foremost killer disease in Nigeria, with an estimated 300,000 children dying from it each year. It accounts for over 25% of infant mortality (children under one year old), 30% of childhood mortality (children under five years old), and 11% of maternal mortality. At least 50% of the population experiences at least one episode of malaria annually, with children under five having 2 to 4 attacks each year. Malaria is particularly severe among pregnant women and young children due to their relatively lower levels of immunity.

To address malaria, the Society for Family Health (SFH) focuses on treatment and prevention through Pre-Packaged Therapy (PPT) and Long Lasting Insecticide Treated Nets (LLINs). The Federal Ministry of Health has implemented a new treatment policy that includes the use of Artemisinin-based Combination Therapy (ACT) as the first-line drug for treating uncomplicated malaria. SFH also promotes gov

In [20]:
# Same session_id → same history
q21 = 'What about the sixth one.?'
a21 = ask(q21, session_id)

print('👤 User:', q21)
print('🤖 Assistant:', a21)
print('-' * 70)

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


👤 User: What about the sixth one.?
🤖 Assistant: Protein-energy malnutrition accounts for 4% of the top causes of death in Nigeria.
----------------------------------------------------------------------


In [27]:
# Same session_id → same history
q22 = 'Tell me more about the seventh one.'
a22 = ask(q22, session_id)

print('👤 User:', q22)
print('🤖 Assistant:', a22)
print('-' * 70)

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


👤 User: Tell me more about the seventh one.
🤖 Assistant: The context does not provide specific details regarding cancer as a cause of death in Nigeria, other than mentioning it as one of the top causes of death. It states that cancer is included in the list of diseases with a rising burden in Nigeria, but does not provide statistics or specific mortality rates related to cancer.
----------------------------------------------------------------------


In [14]:
# Fresh session for a different topic
session_crop = 'rerank-test-crop'
q3 = 'What are the common crop diseases and how are they controlled?'
a3 = ask(q3, session_crop)

print('👤 User:', q3)
print('🤖 Assistant:', a3)
print('-' * 70)

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


👤 User: What are the common crop diseases and how are they controlled?
🤖 Assistant: The common crop diseases and their control methods are:

1. **Cassava Mosaic Disease**
   - **Affected Crop:** Cassava
   - **Symptoms:** Yellowing and mottling of leaves, stunted growth, reduced yield.
   - **Control/Cure:** Use disease-free cuttings, plant resistant varieties, and remove infected plants early.

2. **Maize Smut**
   - **Affected Crop:** Maize
   - **Symptoms:** Large grey or black galls on ears, stalks, and leaves.
   - **Control/Cure:** Remove and destroy infected plants, rotate crops, and plant resistant hybrids.

3. **Rice Blast**
   - **Affected Crop:** Rice
   - **Symptoms:** Spindle-shaped lesions on leaves, panicle damage, and reduced grain quality.
   - **Control/Cure:** Use resistant varieties, avoid excessive nitrogen fertiliser, and apply recommended fungicides.

4. **Tomato Leaf Curl Virus**
   - **Affected Crop:** Tomatoes and peppers
   - **Symptoms:** Curling and yellowi

In [15]:
# New sessions for each topic

# Q1: Follow-up on crops (tests rewrite + reranker together)
session1 = 'diverse-crops'
a1 = ask('What are the common crop diseases?', session1)
print('👤 Q1: What are the common crop diseases?')
print('🤖 A1:', a1[:250])
print('-' * 70)

# Follow-up on the same session
a2 = ask('How do I control the second and the last one?', session1)
print('👤 Q2: How do I control the second and the last one?')
print('🤖 A2:', a2[:250])
print('-' * 70)

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


👤 Q1: What are the common crop diseases?
🤖 A1: The common crop diseases are:

1. Cassava Mosaic Disease
2. Maize Smut
3. Rice Blast
4. Tomato Leaf Curl Virus
----------------------------------------------------------------------


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


👤 Q2: How do I control the second and the last one?
🤖 A2: To control Maize Smut, you should remove and destroy infected plants, rotate crops, and plant resistant hybrids. 

To control Tomato Leaf Curl Virus, you should control whitefly vectors, remove infected plants, and use resistant varieties.
----------------------------------------------------------------------


In [16]:
# Livestock question in a fresh session
session_live = 'diverse-livestock'
a3 = ask('What are common livestock diseases and their controls?', session_live)
print('👤 Q: What are common livestock diseases and their controls?')
print('🤖 A:', a3[:500])
print('-' * 70)

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


👤 Q: What are common livestock diseases and their controls?
🤖 A: Common livestock diseases and their controls are:

1. **Newcastle Disease**
   - Affected Animal: Poultry (chickens, turkeys)
   - Symptoms: Respiratory distress, greenish diarrhoea, nervous signs, high mortality.
   - Control: Vaccinate birds regularly, maintain hygiene, and quarantine new birds.

2. **Foot and Mouth Disease**
   - Affected Animal: Cattle, sheep, goats, pigs
   - Symptoms: Blisters in mouth and hooves, fever, lameness, excessive salivation.
   - Control: Vaccination, movement c
----------------------------------------------------------------------


Precise statistics question

In [17]:
# Test precise number retrieval
session_stats = 'diverse-stats'
a4 = ask('What percentage of deaths in Nigeria are caused by malaria?', session_stats)
print('👤 Q: What percentage of deaths in Nigeria are caused by malaria?')
print('🤖 A:', a4[:400])
print('-' * 70)

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


👤 Q: What percentage of deaths in Nigeria are caused by malaria?
🤖 A: Malaria accounts for 20% of deaths in Nigeria.
----------------------------------------------------------------------


Cross‑topic question

In [18]:
# Cross-topic question
session_cross = 'diverse-cross'
a5 = ask('How does agriculture affect health in Nigeria?', session_cross)
print('👤 Q: How does agriculture affect health in Nigeria?')
print('🤖 A:', a5[:500])
print('-' * 70)

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


👤 Q: How does agriculture affect health in Nigeria?
🤖 A: The provided context does not directly address how agriculture affects health in Nigeria. Therefore, I don't know the answer to that question based on the given information.
----------------------------------------------------------------------


Out‑of‑domain question

In [19]:
# Out-of-domain question
session_unknown = 'diverse-unknown'
a6 = ask('What is the capital of France?', session_unknown)
print('👤 Q: What is the capital of France?')
print('🤖 A:', a6[:200])
print('-' * 70)

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


👤 Q: What is the capital of France?
🤖 A: I don't know.
----------------------------------------------------------------------


In [23]:
# Check what the rewriter produced
rewritten = rewrite_question('Tell me more about the sixth one.', store['rerank-test-1'])
print('Rewritten:', rewritten)

# Check what documents were retrieved
docs = reranking_retriever.invoke(rewritten)
for i, d in enumerate(docs):
    print(f'--- {i} (score={d.metadata.get("relevance_score", "n/a")}) ---')
    print(d.page_content[:250])
    print()

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Rewritten: Tell me more about protein-energy malnutrition.


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


--- 0 (score=0.03396269679069519) ---
 Malaria (20%) 
 Lower Respiratory Infection (19%) 
 HIV/AIDS (9%) 
 Diarrheal Diseases (5%) 
 Road Injuries (5%) 
 Protein Energy Malnutrition (4%) 
 Cancer (3%) 
 Meningitis (3%) 
 Stroke (3%) 
 Tuberculosis (2%) 
 
Public health Issues 


--- 1 (score=0.015030398033559322) ---
 The top causes of death in Nigeria are; malaria, 
lower respiratory infections, HIV/AIDS,      
diarrheal diseases, road injuries, protein -energy 
malnutrition, cancer, meningitis, stroke and 
tuberculosis. 
 Malaria remains the foremost killer d

--- 2 (score=1.7792834114516154e-05) ---
Introduction
O Crop diseases are those conditions 
when pathogens, lack or inadequacy 
of nutrient supply and various other 
factors of the environment cause 
disruptions or hinderances to the 
normal and natural processes of plant 
growth, developme

--- 3 (score=1.641765993554145e-05) ---
Muhammad et al.   Public health problems in Nigeria  
disease, as well as the rapid a